# Test PaliGemma 2-3B Mix 448 — Detección de escaleras

**TFG VisionLeader UFV**

Sube una imagen, ajusta el prompt en la celda correspondiente y ejecuta para ver si el modelo detecta escaleras (u otros objetos).

- **Modelo:** `google/paligemma2-3b-mix-448`
- **Uso:** Ejecuta las celdas en orden. En la celda "Prompt" puedes cambiar el texto manualmente y volver a ejecutar la celda de inferencia.

## 1. Instalar dependencias

In [ ]:
!pip install -q "transformers>=4.40.0" accelerate pillow huggingface_hub

## 2. Login Hugging Face

Necesitas un token con acceso a PaliGemma: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import login
login()  # Pide el token la primera vez

## 3. Cargar modelo PaliGemma 2-3B Mix 448

In [ ]:
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

MODEL_ID = "google/paligemma2-3b-mix-448"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print(f"Cargando {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if DEVICE == "cuda" else None,
    low_cpu_mem_usage=True,
)
if DEVICE == "cpu":
    model = model.to(DEVICE)
model.eval()
print("Modelo cargado.")

## 4. Subir / elegir imagen

- **En Colab:** se usará `files.upload()` para elegir un archivo.
- **En Jupyter local:** puedes poner la ruta en `IMAGE_PATH` (ej: `r"C:\Users\...\stairs.jpg"`).

In [ ]:
from PIL import Image
import io

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
    image = Image.open(image_path).convert("RGB")
    print(f"Imagen cargada: {image_path}")
else:
    # En local: cambia esta ruta por tu imagen
    IMAGE_PATH = "building_sign/ufv_staircase_06.jpg"  # o la ruta que quieras
    try:
        image = Image.open(IMAGE_PATH).convert("RGB")
        print(f"Imagen cargada: {IMAGE_PATH}")
    except FileNotFoundError:
        print("No se encontró la imagen. Cambia IMAGE_PATH o sube un archivo en Colab.")
        image = None

if image is not None:
    display(image)

## 5. Prompt (cámbialo aquí y vuelve a ejecutar la celda de inferencia)

Edita la variable `PROMPT` para probar distintas preguntas (detección de escaleras, ascensor, cartel, etc.).

In [ ]:
# Cambia este prompt manualmente para probar si detecta escaleras u otros objetos
PROMPT = "Is there a staircase visible in this image? Answer with one word: yes or no."

# Otros ejemplos que puedes probar:
# PROMPT = "What object is visible in this image? Answer with one word: elevator, stairs, sign, door, or other."
# PROMPT = "Is there an elevator visible in this image? Answer with one word: yes or no."
# PROMPT = "What do you see in this image? Answer with one word."

print("Prompt actual:")
print(PROMPT)

## 6. Inferencia: preguntar al modelo

In [ ]:
if image is None:
    print("No hay imagen cargada. Ejecuta la celda 4 primero.")
else:
    if not PROMPT.strip().startswith("<image>"):
        text_input = "<image>\n" + PROMPT
    else:
        text_input = PROMPT

    inputs = processor(
        text=text_input,
        images=image,
        padding="longest",
        return_tensors="pt",
    ).to(model.device)
    inputs = inputs.to(dtype=model.dtype)

    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    response = processor.decode(output[0][input_len:], skip_special_tokens=True).strip()
    print("Respuesta del modelo:")
    print(response)